# Medicine Similarity Recommendation

## Objective

The objective of this notebook is to build a content-based
medicine similarity system using medicine information.

The system uses medicine composition, uses, and side-effect
information to identify medicines with similar textual profiles.

The recommendation component uses:

- Text preprocessing
- TF-IDF vectorization
- Cosine similarity
- Medicine similarity retrieval

In [15]:
# Load medicine details dataset

import pandas as pd

medicine_df = pd.read_csv("../data/Medicine_Details.csv")

print("Dataset shape:", medicine_df.shape)
print("\nColumns:")
print(medicine_df.columns.tolist())

Dataset shape: (11825, 9)

Columns:
['Medicine Name', 'Composition', 'Uses', 'Side_effects', 'Image URL', 'Manufacturer', 'Excellent Review %', 'Average Review %', 'Poor Review %']


In [16]:
# Check missing values

missing_values = medicine_df.isnull().sum()

print(missing_values)
print("\nTotal missing values:", missing_values.sum())

Medicine Name         0
Composition           0
Uses                  0
Side_effects          0
Image URL             0
Manufacturer          0
Excellent Review %    0
Average Review %      0
Poor Review %         0
dtype: int64

Total missing values: 0


In [17]:
# Check duplicate medicine records

duplicate_count = medicine_df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 84


In [18]:
# Remove duplicate records and reset the index

medicine_df_clean = (
    medicine_df
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

print("Original rows:", len(medicine_df))
print("Rows after removing duplicates:", len(medicine_df_clean))
print("Duplicates removed:", len(medicine_df) - len(medicine_df_clean))

Original rows: 11825
Rows after removing duplicates: 11741
Duplicates removed: 84


In [19]:
# Create combined text for medicine similarity

medicine_df_clean["combined_text"] = (
    medicine_df_clean["Medicine Name"].astype(str) + " " +
    medicine_df_clean["Composition"].astype(str) + " " +
    medicine_df_clean["Uses"].astype(str) + " " +
    medicine_df_clean["Side_effects"].astype(str)
)

print(medicine_df_clean[[
    "Medicine Name",
    "combined_text"
]].head())

              Medicine Name                                      combined_text
0   Avastin 400mg Injection  Avastin 400mg Injection Bevacizumab (400mg)  C...
1  Augmentin 625 Duo Tablet  Augmentin 625 Duo Tablet Amoxycillin  (500mg) ...
2       Azithral 500 Tablet  Azithral 500 Tablet Azithromycin (500mg) Treat...
3          Ascoril LS Syrup  Ascoril LS Syrup Ambroxol (30mg/5ml) + Levosal...
4         Aciloc 150 Tablet  Aciloc 150 Tablet Ranitidine (150mg) Treatment...


In [20]:
# Convert medicine text into TF-IDF vectors

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(
    medicine_df_clean["combined_text"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (11741, 9248)


In [21]:
# Calculate cosine similarity between medicines

from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim.shape)

Cosine similarity matrix shape: (11741, 11741)


In [22]:
# Create medicine similarity recommendation function

def recommend_medicines(medicine_name, top_n=5):
    if medicine_name not in medicine_df_clean["Medicine Name"].values:
        return f"Medicine '{medicine_name}' not found."

    medicine_index = medicine_df_clean[
        medicine_df_clean["Medicine Name"] == medicine_name
    ].index[0]

    similarity_scores = list(
        enumerate(cosine_sim[medicine_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similar_indices = [
        index for index, score in similarity_scores[1:top_n + 1]
    ]

    recommendations = medicine_df_clean.iloc[
        similar_indices
    ][[
        "Medicine Name",
        "Composition",
        "Uses",
        "Side_effects"
    ]].copy()

    return recommendations

In [23]:
# Medicine similarity recommendation with similarity scores

def recommend_medicines(medicine_name, top_n=5):
    if medicine_name not in medicine_df_clean["Medicine Name"].values:
        return f"Medicine '{medicine_name}' not found."

    medicine_index = medicine_df_clean[
        medicine_df_clean["Medicine Name"] == medicine_name
    ].index[0]

    similarity_scores = list(
        enumerate(cosine_sim[medicine_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_similar = similarity_scores[1:top_n + 1]

    recommendations = medicine_df_clean.iloc[
        [index for index, score in top_similar]
    ][[
        "Medicine Name",
        "Composition",
        "Uses",
        "Side_effects"
    ]].copy()

    recommendations["Similarity Score"] = [
        score for index, score in top_similar
    ]

    return recommendations

In [24]:
# Test recommendation with similarity scores

test_medicine = "Azeflo Nasal Spray"

recommendations = recommend_medicines(
    test_medicine,
    top_n=5
)

print("Input medicine:", test_medicine)
print("\nSimilar medicines:")
recommendations

Input medicine: Azeflo Nasal Spray

Similar medicines:


,Medicine Name,Composition,Uses,Side_effects,Similarity Score
6168,Nazomac-AF Nasal Spray,Fluticasone Propionate (50mcg) + Azelastine (1...,Treatment of Sneezing and runny nose due to al...,Taste change Nosebleeds Headache Cough Upper r...,0.897362
462,Azeflo FT Nasal Spray,Fluticasone Propionate (27.5mcg) + Azelastine ...,Treatment of Sneezing and runny nose due to al...,Taste change Nosebleeds Headache Cough Upper r...,0.896814
1928,Ezicas AZ Nasal Spray,Fluticasone Propionate (50mcg) + Azelastine (1...,Treatment of Sneezing and runny nose due to al...,Taste change Nosebleeds Headache Cough Upper r...,0.883072
2429,Flublock-AZ Nasal Spray,Fluticasone Propionate (50mcg) + Azelastine (1...,Treatment of Sneezing and runny nose due to al...,Taste change Nosebleeds Headache Cough Upper r...,0.880917
9321,Seroflo 50 Inhaler,Salmeterol (25mcg) + Fluticasone Propionate (5...,Treatment of Chronic obstructive pulmonary dis...,Headache Cough Upper respiratory tract infecti...,0.575507


## Interpretation

The content-based recommendation system identifies medicines
with similar textual profiles based on their medicine name,
composition, uses, and side-effect information.

TF-IDF converts the medicine information into numerical vectors,
and cosine similarity measures the similarity between these vectors.

A higher similarity score indicates greater textual similarity
between two medicine records.

The similarity score should not be interpreted as clinical
equivalence, treatment effectiveness, safety, or a recommendation
to substitute one medicine for another.

## Conclusion

The medicine similarity module successfully implements a
content-based filtering approach using medicine information.

The system:

- Uses medicine name, composition, uses, and side-effect information.
- Removes duplicate medicine records before processing.
- Converts medicine text into TF-IDF vectors.
- Calculates cosine similarity between medicine profiles.
- Returns the top similar medicines with similarity scores.

The module demonstrates how content-based filtering can be used
to identify medicines with similar textual characteristics.

This module is intended for educational and informational purposes.
The similarity results do not represent medical advice or confirm
that medicines are clinically equivalent or interchangeable.